# RetailMart Lakehouse

## Notebook : 02_Load_Orders

### Layer
Bronze Layer

### Objective

This notebook ingests the Orders dataset from the Raw Volume into the Bronze layer.

The notebook performs:

- Read Raw Orders CSV
- Initial Data Profiling
- Data Validation
- Audit Column Addition
- Delta Table Creation
- Post Load Verification

### Source

Raw Volume

### Target

retailmart.bronze.orders

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import *

from datetime import datetime
import uuid

In [0]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])

In [0]:
SOURCE_FILE = RAW_ORDERS
TARGET_TABLE = TARGET_TABLE_ORDERS
PIPELINE_NAME = "Bronze_Orders"
RUN_ID = generate_run_id()
START_TIME = start_pipeline()

In [0]:
orders_df = (
    spark.read
         .schema(orders_schema)
         .option("header", True)
         .csv(SOURCE_FILE)
)

In [0]:
total_rows = orders_df.count()
total_columns = len(orders_df.columns)
print("ORDERS DATASET PROFILE")
print(f"Rows    : {total_rows}")
print(f"Columns : {total_columns}")
orders_df.printSchema()

ORDERS DATASET PROFILE
Rows    : 50000
Columns : 8
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
display(orders_df.limit(10))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
ORD_0000001,CUST_006571,delivered,2023-06-15T14:30:00.000Z,2023-06-16T12:30:00.000Z,2023-06-17T12:30:00.000Z,2023-06-19T14:30:00.000Z,2023-06-29T14:30:00.000Z
ORD_0000002,CUST_006956,cancelled,2021-06-12T11:02:00.000Z,2021-06-12T23:02:00.000Z,2021-06-13T23:02:00.000Z,null,2021-06-21T11:02:00.000Z
ORD_0000003,CUST_008373,delivered,2022-03-31T19:38:00.000Z,2022-04-01T04:38:00.000Z,2022-04-04T04:38:00.000Z,2022-04-14T19:38:00.000Z,2022-04-17T19:38:00.000Z
ORD_0000004,CUST_007986,delivered,2021-09-09T22:29:00.000Z,2021-09-10T11:29:00.000Z,2021-09-13T11:29:00.000Z,2021-09-23T22:29:00.000Z,2021-09-27T22:29:00.000Z
ORD_0000005,CUST_002810,shipped,2022-08-27T07:46:00.000Z,2022-08-28T03:46:00.000Z,2022-08-29T03:46:00.000Z,null,2022-09-15T07:46:00.000Z
ORD_0000006,CUST_007867,invoiced,2023-05-26T16:27:00.000Z,2023-05-26T22:27:00.000Z,2023-05-29T22:27:00.000Z,null,2023-06-09T16:27:00.000Z
ORD_0000007,CUST_004827,processing,2021-09-21T15:50:00.000Z,2021-09-22T11:50:00.000Z,2021-09-23T11:50:00.000Z,null,2021-09-28T15:50:00.000Z
ORD_0000008,CUST_005137,delivered,2021-01-19T14:24:00.000Z,2021-01-20T10:24:00.000Z,2021-01-22T10:24:00.000Z,2021-01-28T14:24:00.000Z,2021-02-08T14:24:00.000Z
ORD_0000009,CUST_008418,delivered,2021-07-07T23:18:00.000Z,2021-07-08T23:18:00.000Z,2021-07-10T23:18:00.000Z,2021-07-20T23:18:00.000Z,2021-07-21T23:18:00.000Z
ORD_0000010,CUST_011541,shipped,2021-02-21T04:01:00.000Z,2021-02-21T05:01:00.000Z,2021-02-23T05:01:00.000Z,null,2021-03-01T04:01:00.000Z


In [0]:
expected_schema = {
    "order_id": "string",
    "customer_id": "string",
    "order_status": "string",
    "order_purchase_timestamp": "timestamp",
    "order_approved_at": "timestamp",
    "order_delivered_carrier_date": "timestamp",
    "order_delivered_customer_date": "timestamp",
    "order_estimated_delivery_date": "timestamp"
}

In [0]:
schema_status = validate_schema(
    orders_df,
    expected_schema
)

Schema Validation Passed


In [0]:
pk_status = validate_primary_key(
    orders_df,
    "order_id"
)

Total Rows : 50000
Distinct Count : 50000
Primary Key Validation Passed — (order_id)


# Duplicate Validation

In [0]:
duplicate_rows = duplicate_summary(orders_df,total_rows)

Duplicate Rows : 0


# Null Value Analysis

In [0]:
null_summary(orders_df)

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,0,0,0,0,0,25034,0


In [0]:
# Business Validation

orders_df.groupBy("order_status") \
         .count() \
         .orderBy(F.desc("count")) \
         .show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|24966|
|    invoiced| 6421|
|     shipped| 6288|
|   cancelled| 6193|
|  processing| 6132|
+------------+-----+



In [0]:
# Purchase Timestamp null check
null_purchase = orders_df.filter(
    col("order_purchase_timestamp").isNull()
).count()

print(f"Orders with NULL Purchase Timestamp : {null_purchase}")

Orders with NULL Purchase Timestamp : 0


In [0]:
# Estimate delivery validation 
invalid_delivery = orders_df.filter(
    col("order_estimated_delivery_date") <
    col("order_purchase_timestamp")
).count()

print(f"Invalid Estimated Delivery Dates : {invalid_delivery}")

Invalid Estimated Delivery Dates : 0


In [0]:
orders_df.describe().show()

+-------+-----------+-----------+------------+
|summary|   order_id|customer_id|order_status|
+-------+-----------+-----------+------------+
|  count|      50000|      50000|       50000|
|   mean|       NULL|       NULL|        NULL|
| stddev|       NULL|       NULL|        NULL|
|    min|ORD_0000001|CUST_000001|   cancelled|
|    max|ORD_0050000|CUST_015000|     shipped|
+-------+-----------+-----------+------------+



# Audit columns

In [0]:
orders_df = add_audit_columns(orders_df,PIPELINE_NAME,RUN_ID)

try:
    (
        orders_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )
    status = "SUCCESS"
    print("Bronze Table Created Successfully")

except Exception as err:
    status = "FAILED"
    print(f"Error: {err}")

In [0]:
status, error = write_bronze_table(orders_df,TARGET_TABLE)

Bronze table written: retailmart.bronze.orders


In [0]:
# Verification
bronze_df = spark.table(TARGET_TABLE)
rows_written = bronze_df.count()
print(f"Rows Written : {rows_written}")
display(bronze_df.limit(5))

Rows Written : 50000


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,ingestion_date,pipeline_name,run_id
ORD_0000001,CUST_006571,delivered,2023-06-15T14:30:00.000Z,2023-06-16T12:30:00.000Z,2023-06-17T12:30:00.000Z,2023-06-19T14:30:00.000Z,2023-06-29T14:30:00.000Z,2026-07-14T18:00:48.029Z,2026-07-14,Bronze_Orders,5ac5ab34-f966-4a45-b0f0-244882b806f9
ORD_0000002,CUST_006956,cancelled,2021-06-12T11:02:00.000Z,2021-06-12T23:02:00.000Z,2021-06-13T23:02:00.000Z,null,2021-06-21T11:02:00.000Z,2026-07-14T18:00:48.029Z,2026-07-14,Bronze_Orders,5ac5ab34-f966-4a45-b0f0-244882b806f9
ORD_0000003,CUST_008373,delivered,2022-03-31T19:38:00.000Z,2022-04-01T04:38:00.000Z,2022-04-04T04:38:00.000Z,2022-04-14T19:38:00.000Z,2022-04-17T19:38:00.000Z,2026-07-14T18:00:48.029Z,2026-07-14,Bronze_Orders,5ac5ab34-f966-4a45-b0f0-244882b806f9
ORD_0000004,CUST_007986,delivered,2021-09-09T22:29:00.000Z,2021-09-10T11:29:00.000Z,2021-09-13T11:29:00.000Z,2021-09-23T22:29:00.000Z,2021-09-27T22:29:00.000Z,2026-07-14T18:00:48.029Z,2026-07-14,Bronze_Orders,5ac5ab34-f966-4a45-b0f0-244882b806f9
ORD_0000005,CUST_002810,shipped,2022-08-27T07:46:00.000Z,2022-08-28T03:46:00.000Z,2022-08-29T03:46:00.000Z,null,2022-09-15T07:46:00.000Z,2026-07-14T18:00:48.029Z,2026-07-14,Bronze_Orders,5ac5ab34-f966-4a45-b0f0-244882b806f9


In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_FILE,
    target=TARGET_TABLE,
    rows_read=total_rows,
    rows_written=rows_written,
    duplicate_count=duplicate_rows,
    start_time=START_TIME,
    status=status
)

BRONZE LOAD REPORT
Pipeline        : Bronze_Orders
Run ID          : 5ac5ab34-f966-4a45-b0f0-244882b806f9
Source          : /Volumes/dbacademy/default/raw/raw_orders_dataset.csv
Target          : retailmart.bronze.orders
Rows Read       : 50000
Rows Written    : 50000
Duplicate Rows  : 0
Start Time      : 2026-07-14 18:00:35.836744
End Time        : 2026-07-14 18:00:51.575686
Duration (sec)  : 15.74
Status          : SUCCESS
Error:          : None
